# Qwen3-8B QLoRA SFT - Chess Logic GPT

Fine-tunes Qwen3-8B with 4-bit QLoRA on calc-trace chess reasoning plus formal logic and memory data.

**Run mode:** one notebook run adds one 500-step training window. If a Google Drive checkpoint exists, the notebook restores it and sets `max_steps = last_checkpoint_step + 500`, so rerunning keeps extending training instead of stopping at global step 500.

**Runtime:** Colab T4 GPU. The trainer automatically uses fp16 on T4/P100-class GPUs and bf16 only on GPUs that actually support it.


In [ ]:
#@title 0. Keep-alive watchdog
#@markdown Run this first, then keep the tab open and keep your laptop awake.
from IPython.display import Javascript, display

JS_KEEPALIVE = "\n(() => {\n  if (window._colabKeepAlive) return;\n  window._colabKeepAlive = true;\n  setInterval(() => {\n    document.dispatchEvent(new MouseEvent('mousemove', {clientX: 1, clientY: 1}));\n    document.dispatchEvent(new KeyboardEvent('keydown', {key: 'Shift', keyCode: 16}));\n    document.dispatchEvent(new KeyboardEvent('keyup', {key: 'Shift', keyCode: 16}));\n  }, 180000);\n  console.log('Colab keep-alive watchdog started');\n})();\n"

display(Javascript(JS_KEEPALIVE))
print('Watchdog injected. Keep the browser tab open and prevent laptop sleep.')


In [ ]:
#@title 1. Install dependencies + verify GPU
!pip install -q --upgrade-strategy only-if-needed \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    "transformers>=4.51" "peft>=0.12" "accelerate>=0.33" \
    "datasets>=2.20" trackio python-chess orjson pyyaml zstandard huggingface_hub
!pip install -q --no-deps bitsandbytes

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
else:
    raise RuntimeError('No GPU. Runtime > Change runtime type > T4 GPU')


In [ ]:
#@title 2. Download training data from HuggingFace
from huggingface_hub import hf_hub_download
import os

LOCAL_DATA = '/content/data/processed'
os.makedirs(LOCAL_DATA, exist_ok=True)

DATA_REPO = 'Tobiasd2/chess-logic-gpt-sft-data'
for name in ('train_mix.jsonl', 'eval_mix.jsonl', 'eval_puzzles_ood.jsonl'):
    dst = f'{LOCAL_DATA}/{name}'
    if not os.path.exists(dst):
        print(f'Downloading {name}...')
        hf_hub_download(repo_id=DATA_REPO, filename=name, repo_type='dataset', local_dir=LOCAL_DATA)
    else:
        print(f'Already have {name}')

print('Data ready:', sorted(os.listdir(LOCAL_DATA)))


In [ ]:
#@title 3. Clone latest repo & install package
import os

REPO_DIR = '/content/chess-logic-gpt'
REPO_URL = 'https://github.com/tdickers22-lgtm/chess-logic-gpt.git'
BRANCH = 'main'

if not os.path.isdir(f'{REPO_DIR}/.git'):
    !rm -rf {REPO_DIR}
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

!pip install -q --no-deps -e {REPO_DIR}
print('Package installed from latest main')


In [ ]:
#@title 4. Build config + restore checkpoint from Drive
import os
import re
import shutil
import yaml
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = '/content/outputs/qwen3-8b-chess-logic-lora'
SESSION_STEPS = 500
os.makedirs(OUTPUT_DIR, exist_ok=True)

def checkpoint_step(path):
    if not path:
        return 0
    match = re.search(r'checkpoint-(\d+)$', os.path.basename(path.rstrip('/')))
    return int(match.group(1)) if match else 0

CKPT_DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CKPT_DRIVE = '/content/drive/MyDrive/chess-logic-gpt/checkpoints'
    if os.path.isdir(CKPT_DRIVE):
        drive_ckpt = get_last_checkpoint(CKPT_DRIVE)
        if drive_ckpt:
            ckpt_name = os.path.basename(drive_ckpt)
            local_ckpt = f'{OUTPUT_DIR}/{ckpt_name}'
            if not os.path.isdir(local_ckpt):
                shutil.copytree(drive_ckpt, local_ckpt)
                print(f'Restored from Drive: {ckpt_name}')
            else:
                print(f'Already local: {ckpt_name}')
        else:
            print('No checkpoint on Drive - starting fresh')
    else:
        print('No checkpoint folder on Drive - starting fresh')
except Exception as e:
    print(f'Drive not available ({e}) - starting fresh')

local_last = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
prior_step = checkpoint_step(local_last)
max_steps = max(SESSION_STEPS, prior_step + SESSION_STEPS)
print(f'Last checkpoint step: {prior_step}; this run target max_steps: {max_steps}')

config = {
    'model': {
        'base_model': 'Qwen/Qwen3-8B',
        'trust_remote_code': True,
        'load_in_4bit': True,
        'bnb_4bit_compute_dtype': 'auto',
        'enable_thinking': False,
    },
    'data': {
        'train_file': '/content/data/processed/train_mix.jsonl',
        'eval_file': '/content/data/processed/eval_mix.jsonl',
        'text_field': 'text',
        'max_seq_length': 1024,
    },
    'lora': {
        'r': 32,
        'alpha': 64,
        'dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                           'gate_proj', 'up_proj', 'down_proj'],
    },
    'training': {
        'output_dir': OUTPUT_DIR,
        'max_steps': max_steps,
        'per_device_train_batch_size': 1,
        'per_device_eval_batch_size': 1,
        'gradient_accumulation_steps': 16,
        'learning_rate': 8e-5,
        'warmup_ratio': 0.03,
        'weight_decay': 0.01,
        'logging_steps': 10,
        'eval_steps': 50,
        'save_steps': 25,
        'save_total_limit': 5,
        'checkpoint_mirror_dir': CKPT_DRIVE,
        'checkpoint_mirror_keep': 5,
        'bf16': 'auto',
        'fp16': 'auto',
        'gradient_checkpointing': True,
        'resume_from_checkpoint': True,
        'report_to': 'none',
    },
}

with open('/content/train_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config ready:', '/content/train_config.yaml')


In [ ]:
#@title 5. Train
import os
import time

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
START = time.time()

!cd /content/chess-logic-gpt && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True TOKENIZERS_PARALLELISM=false python scripts/train_lora.py --config /content/train_config.yaml

elapsed = (time.time() - START) / 3600
print(f'\nTraining took {elapsed:.2f} hours')


In [ ]:
#@title 6. Save latest checkpoint to Drive
import glob as globlib
import os
import re as regex
import shutil

OUTPUT_DIR = '/content/outputs/qwen3-8b-chess-logic-lora'
CKPT_DRIVE = globals().get('CKPT_DRIVE')

def checkpoint_step(path):
    match = regex.search(r'checkpoint-(\d+)$', os.path.basename(path.rstrip('/')))
    return int(match.group(1)) if match else -1

ckpts = sorted(globlib.glob(f'{OUTPUT_DIR}/checkpoint-*'), key=checkpoint_step)

if ckpts and CKPT_DRIVE:
    os.makedirs(CKPT_DRIVE, exist_ok=True)
    latest = ckpts[-1]
    ckpt_name = os.path.basename(latest)
    drive_dst = f'{CKPT_DRIVE}/{ckpt_name}'
    if os.path.exists(drive_dst):
        shutil.rmtree(drive_dst)
    shutil.copytree(latest, drive_dst)

    drive_ckpts = sorted(globlib.glob(f'{CKPT_DRIVE}/checkpoint-*'), key=checkpoint_step)
    for old in drive_ckpts[:-5]:
        shutil.rmtree(old)
    print(f'Saved to Drive: {ckpt_name}')
elif ckpts:
    print(f'Checkpoint available at {ckpts[-1]} (Drive not mounted, skipping upload)')
else:
    print('No checkpoints found')

print('Done. Re-run this notebook to continue with another 500-step window.')
